# Emergence and stability of depth responses

Analysis of depth responses in neurons at day 1 to 5

In [ ]:
from v1_depth_map.revisions.revision_sessions import sessions
# Unfortunately the data for Day 1 of PZAH is missing (only 20 minutes of recording
# left, currently with synchronisation issue). Exclude the mouse

sphere_sessions = [(k,v) for k,v in sessions.items() if ('sphere' in v)]
mice = set([sess[0].split('_')[0] for sess in sphere_sessions])
print(f'{len(mice)} mice')
print(mice)

In [ ]:
# Load all neurons_df
import numpy as np
import flexiznam as flz
import pandas as pd
import matplotlib as mpl


In [ ]:


project = 'colasa_3d-vision_revisions'
save_folder = flz.get_processed_path(
        "colasa_3d-vision_revisions/analysis/multi_days"
    )
save_folder.mkdir(exist_ok=True)
neurons_dfs = dict()
for sess_name, protocol in sphere_sessions:
    mouse, sess = sess_name.split('_')
    neurons_df = pd.read_pickle(flz.get_processed_path(project) / mouse / sess / 'neurons_df.pickle')
    neurons_df['mouse'] = mouse
    neurons_df['session'] = sess_name
    neurons_df['uid'] = neurons_df.session.astype(str) + '_' + (neurons_df.roi).astype(str) 
    neurons_df['day'] = int(protocol.split('_')[1])
    neurons_dfs[sess_name] =neurons_df
neurons_df = pd.concat(neurons_dfs.values(), ignore_index=True)
print(f'Loaded {len(neurons_df)} neurons from {len(neurons_df.session.unique())} sessions')


In [ ]:
# Add depth tuned boolean
neurons_df["depth_tuned"] = neurons_df.apply(
    lambda x: x["depth_tuning_test_spearmanr_rval_closedloop"] > 0.1
    and x["depth_tuning_test_spearmanr_pval_closedloop"] < 0.05,
    axis=1,
)

In [ ]:
# Unfortunately the day 1 of PZAH17.1e crashed. We have half a recording and the sync
# between vis stim and recording is broken, so results are meaningless
to_remove = neurons_df[(neurons_df.mouse == 'PZAH17.1e') & (neurons_df.day == 1)].index
print(f'Removing {len(to_remove)} neurons from invalid dataset')
neurons_df = neurons_df.drop(index=to_remove)


In [ ]:
import matplotlib.pyplot as plt
from cottage_analysis.plotting import depth_selectivity_plots
fontsize_dict = {"title": 7, "label": 7, "tick": 5, "legend": 5}

ax = plt.subplot(1,1,1)
all_frac = []
for mouse, mouse_df in neurons_df.groupby('mouse'):
    print(mouse)
    frac = np.zeros(5, dtype=float) + np.nan
    for day, day_df in mouse_df.groupby('day'):
        valid = day_df[~np.isnan(day_df.depth_tuned)]
        frac[day-1] = (valid.depth_tuned.sum()/len(valid.depth_tuned))
        ax.text(day-1, frac[day-1]+0.01, len(valid.depth_tuned), fontsize=6, horizontalalignment='center', verticalalignment='bottom')
    ax.plot(np.arange(5), frac, 'o-', label=mouse)
    all_frac.append(frac)
fractions_depth_by_day = np.vstack(all_frac)
ax.legend(loc='lower right')
ax.set_xticks(np.arange(5), labels=[f'Day {i}' for i in range(1,6)])
ax.set_ylabel('Proportion depth tuned')

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
m = np.nanmean(fractions_depth_by_day, axis=0)
std = np.nanstd(fractions_depth_by_day, axis=0)
ax.fill_between(np.arange(5), m-std, m+std, color='k', alpha=0.2)
ax.plot(m, marker='o', color='k')
ax.set_xlim(0, 4)
ax.set_ylim(0,1)
ax.set_xticks(np.arange(5), labels=[f'Day {i}' for i in range(1, 6)])
ax.set_ylabel('Proportion of depth tuned neurons')
_ = ax.spines[['right', 'top']].set_visible(False)

In [ ]:
from pathlib import Path
# import roicat.util
import json

root_path = '/Volumes/lab-znamenskiyp/home/shared/projects/colasa_3d-vision_revisions'
root_path = str(flz.get_processed_path(project))
neurons_df['tracking_id_manual'] = np.nan
neurons_df['tracking_uid_manual'] = ""


for mouse in mice:
    print(f'Loading data for {mouse}')
    roicat_path='/'.join([root_path, mouse, 'ROICat_spheretubes', 
        f"{mouse}_roicat.tracking.params_used.json"])

    with open(roicat_path) as roicat_file:
        roicat_params = json.load(roicat_file)
    session_list = []
    for statfile in roicat_params['data']['__init__']['paths_statFiles']:
        session_list.append(statfile.split(mouse)[1].split('/')[1])

    manual_click = Path('/'.join([root_path, mouse, 'ROICat_spheretubes', 
        'matches.json']))
    
    if manual_click.exists():
        print(f'Found manual click {manual_click}')
        # Load it
        with open(manual_click, 'r') as f:
            matches = json.load(f)
        matches = np.array(matches, dtype=float)
    else:
        print(f"No manual click for {mouse}")
        matches = None
        continue

    for sess_id, sess in enumerate(session_list):
        sess_name = f"{mouse}_{sess}"
        mask = (neurons_df.session==sess_name)

        if matches is None:
            continue
        # Add manual match
        manual_ids = np.arange(matches.shape[0])
        valid = ~np.isnan(matches[:, sess_id])

        valid_rois = (matches[valid, sess_id].astype(int) - 1).astype(str)
        neuron_uids = [sess_name + '_' + roi for roi in valid_rois]
        valid_manual_ids = manual_ids[valid]
        # Create a dictionary mapping the neuron UID to the manual tracking ID
        uid_to_manual_id = dict(zip(neuron_uids, valid_manual_ids))
        
        # Update the tracking_id_manual column for the current session
        mask_sess = (neurons_df.session == sess_name)
        neurons_df.loc[mask_sess, 'tracking_id_manual'] = neurons_df.loc[mask_sess, 'uid'].map(uid_to_manual_id)


neurons_df['tracking_id_manual'] = np.nan_to_num(neurons_df['tracking_id_manual'], nan=-1)
mask = neurons_df.tracking_id_manual != -1 
neurons_df.loc[mask, 'tracking_uid_manual'] = (
    neurons_df.loc[mask, 'mouse'] + 
    neurons_df.loc[mask, 'tracking_id_manual'].astype(int).astype(str)
)

In [ ]:
tracked_neurons = neurons_df[neurons_df['tracking_uid_manual'] != ""]
tracking_ids = tracked_neurons.tracking_uid_manual.unique()

depth_tuning = np.zeros((len(tracking_ids), 5), float)+ np.nan
is_depth_tuned = np.zeros((len(tracking_ids), 5), float)+ np.nan
for itrack, tid in enumerate(tracking_ids):
    ndf = neurons_df[neurons_df.tracking_uid_manual==tid]
    for day, depth, is_tuned in zip(ndf.day, ndf.preferred_depth_closedloop, ndf.depth_tuned):
        depth_tuning[itrack, day-1] = depth
        is_depth_tuned[itrack, day-1] = is_tuned
    

In [ ]:
valid = np.all(np.nan_to_num(is_depth_tuned, nan=1)==1, axis=1)

d =depth_tuning[valid,:]
plt.imshow(np.log(d[np.argsort(d[:,0]),:]), aspect='auto', cmap='cool', interpolation='None')

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

def get_tracking_matrices(df, tracking_column, n_days=5):
    """Helper to create depth tuning and is_tuned matrices for a given tracking ID."""
    tracked_df = df[df[tracking_column] != ""]
    uids = tracked_df[tracking_column].unique()
    
    tuning_mtx = np.zeros((len(uids), n_days), float) * np.nan
    is_tuned_mtx = np.zeros((len(uids), n_days), float) * np.nan
    
    for i, tid in enumerate(uids):
        ndf = df[df[tracking_column] == tid]
        for day, depth, is_tuned in zip(ndf.day, ndf.preferred_depth_closedloop, ndf.depth_tuned):
            tuning_mtx[i, day-1] = depth
            is_tuned_mtx[i, day-1] = is_tuned
    return tuning_mtx, is_tuned_mtx

# 1. Load ROIcat Automatic Tracking IDs (similar to your manual click loop)
neurons_df['tracking_uid_auto'] = ""

for mouse in mice:
    # Path to ROIcat results
    roicat_dir = Path(flz.get_processed_path(project)) / mouse / 'ROICat_spheretubes'
    results_path = roicat_dir / f"{mouse}_roicat.tracking.results_clusters.json"
    params_path = roicat_dir / f"{mouse}_roicat.tracking.params_used.json"
    
    if not results_path.exists():
        continue
        
    with open(results_path) as f: results = json.load(f)
    with open(params_path) as f: params = json.load(f)
    
    labels = np.array(results['labels'])
    session_list = [s.split(mouse)[1].split('/')[1] for s in params['data']['__init__']['paths_statFiles']]
    
    # Assign Auto IDs by splitting the global 'labels' array back into sessions
    current_idx = 0
    for sess in session_list:
        sess_name = f"{mouse}_{sess}"
        mask = neurons_df.session == sess_name
        n_rois = mask.sum()
        
        # Get labels for this session's ROIs
        sess_labels = labels[current_idx : current_idx + n_rois]
        
        # We match by index assuming the order in neurons_df matches the statFiles used in ROICat
        # ROI IDs in ROICat correspond to the order in the input files
        neurons_df.loc[mask, 'tracking_uid_auto'] = [
            f"{mouse}_auto_{label}" if label != -1 else "" for label in sess_labels
        ]
        current_idx += n_rois

# 2. Generate the Figures
tracking_methods = {
    "Manual Click": "tracking_uid_manual",
    "ROIcat Auto": "tracking_uid_auto"
}

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax, (title, col) in zip(axes, tracking_methods.items()):
    d_mtx, is_tuned_mtx = get_tracking_matrices(neurons_df, col)
    
    # User's filtering logic: must be tuned on all days it was detected (treating missing as tuned)
    valid = np.all(np.nan_to_num(is_tuned_mtx, nan=1) == 1, axis=1)
    d_filtered = d_mtx[valid, :]
    
    # Sort by Day 1 preference
    sort_idx = np.argsort(d_filtered[:, 0])
    im = ax.imshow(np.log(d_filtered[sort_idx, :]), aspect='auto', cmap='cool', interpolation='None')
    
    ax.set_title(f"{title} (n={len(d_filtered)})")
    ax.set_xlabel("Day")
    ax.set_xticks(np.arange(5))
    ax.set_xticklabels(np.arange(1, 6))

plt.tight_layout()
plt.show()


In [ ]:
# Cache for trials_df to avoid reloading for every subplot
trials_df_cache = {}


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import flexiznam as flz
from cottage_analysis.analysis import spheres

# --- 1. Setup and selection ---
project = "colasa_3d-vision_revisions"
flexilims_session = flz.get_flexilims_session(project)

# Select neurons tracked on all 5 days (manual tracking)
tracked = neurons_df[neurons_df.tracking_uid_manual != ""]

# Pick 5 example neurons
def get_amp(popt):
    return np.exp(popt[0]) if (isinstance(popt, (list, np.ndarray)) and len(popt)>0) else 0
# Filter neurons for amplitude >= 0.1
# Note: We check 'depth_tuning_popt_closedloop_running' specifically
neurons_df['fit_amp'] = neurons_df['depth_tuning_popt_closedloop_running'].apply(get_amp)

tracked = neurons_df[
    (neurons_df.tracking_uid_manual != "") & 
    (neurons_df.fit_amp >= 0.4)
]
print(f"{len(tracked)} neurons clicked with fit amp above 10per")
day_counts = tracked.groupby('tracking_uid_manual')['day'].nunique()
full_track_ids = day_counts[day_counts == 5].index.tolist()
print(f'Found {len(full_track_ids)} neurons with dff > 0.5 all days')
np.random.seed(45)
example_tids = np.random.choice(full_track_ids, size=11, replace=False)
avg_depths = tracked.groupby('tracking_uid_manual')['preferred_depth_closedloop_running'].mean()
example_tids = avg_depths.loc[example_tids].sort_values().index
example_tids=(example_tids[[1, 6, 9]]).tolist()



In [ ]:

# Plotting parameters from your reference figure
fontsize_dict = {"title": 7, "label": 7, "tick": 5, "legend": 5}
depth_tuning_kwargs = dict(
    plot_fit=True,
    plot_smooth=False,
    linewidth=2.5,
    closed_loop=1,
    fontsize_dict=fontsize_dict,
    markersize=20,
    markeredgecolor='w',
)


In [ ]:
# --- 2. Plotting (Flipped: Rows=Days, Cols=Cells) ---
# I increased the height (figsize 7x10) since 5 days are now vertical
fig, axes = plt.subplots(5, 3, figsize=(8, 7)) 

for i_col, tid in enumerate(example_tids):
    # Get all days for this specific neuron (this will be one column)
    neuron_days = neurons_df[neurons_df.tracking_uid_manual == tid].sort_values('day')
    
    for i_row, day in enumerate(range(1, 6)):
        ax = axes[i_row, i_col]
        plt.sca(ax)
        
        # Get data for this specific day/cell
        row = neuron_days[neuron_days.day == day].iloc[0]
        sess_name = row['session']
        roi_idx = int(row['roi'])
        
        if sess_name not in trials_df_cache:
            print(f"Loading trials_df for {sess_name}...")
            _, trials_df = spheres.sync_all_recordings(
                session_name=sess_name,
                flexilims_session=flexilims_session,
                project=project,
                filter_datasets={"anatomical_only": 3, 'annotated':True},
                recording_type="two_photon",
                protocol_base="SpheresPermTubeReward",
                photodiode_protocol=5,
                return_volumes=True,
            )
            trials_df_cache[sess_name] = trials_df

        neurons_this_day = neurons_df[(neurons_df.session == sess_name)].set_index('roi')        
        
        depth_selectivity_plots.plot_depth_tuning_curve(
            neurons_df=neurons_this_day,
            trials_df=trials_df_cache[sess_name],
            roi=roi_idx,
            rs_thr=0.05,
            still_only=False,
            linecolor="royalblue",
            ax=ax,
            **depth_tuning_kwargs 
        )
        pref_depth = row['preferred_depth_closedloop_running'] # This is in meters
        ax.axvline(np.log(pref_depth),
         color='royalblue', linestyle='--', linewidth=1, alpha=0.7)
        
        # --- Polish labels for the flipped layout ---
        if i_row == 0: 
            ax.set_title(f"Cell {i_col+1}", fontsize=18)
        
        if i_col == 0: 
            ax.set_ylabel(f"Day {day}", fontsize=18)
        else: 
            ax.set_ylabel("")
            
        if i_row == 4: 
            ax.set_xlabel("Depth (cm)", fontsize=18)
            ax.tick_params(axis='x', rotation=45, labelsize=14,)
        else: 
            ax.set_xlabel("")
            ax.set_xticklabels('')

    # --- Synchronize Y-limits PER COLUMN (one cell across days) ---
    col_axes = axes[:, i_col]
    col_y_max = max(ax.get_ylim()[1] for ax in col_axes)
    # Get a clean top tick
    top_tick = np.round(np.floor(col_y_max * 10) / 10, 1) 
    
    for ir_idx, ax in enumerate(col_axes):
        ax.set_ylim(-0.05, col_y_max)
        ax.set_yticks([0, top_tick])
        ax.set_yticklabels([0, top_tick])
        ax.tick_params(axis='both', labelsize=14)
        sns.despine(ax=ax, offset=3, trim=True)
plt.tight_layout()

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
fig.savefig(save_folder / "consecutive_day_exampleneurons.pdf", 
    format='pdf', 
    bbox_inches='tight',   # Automatically crops whitespace bounds
    transparent=False)  



In [ ]:
neurons_df.depth_tuned

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
tracked = neurons_df[(neurons_df.tracking_uid_manual != "") & (neurons_df.depth_tuned)]
# 1. Pivot the data so we have one column per day for each tracked neuron
# We use the 'preferred_depth_closedloop_running' column
df_pivot = tracked.pivot(
    index='tracking_uid_manual', 
    columns='day', 
    values='preferred_depth_closedloop_running'
)

# 2. Collect all pairs of (Day N, Day N+1)
x_vals = []
y_vals = []

# Iterate through consecutive days (1-2, 2-3, 3-4, 4-5)
rois = set()
for day in range(1, 5):
    next_day = day + 1
    if day in df_pivot.columns and next_day in df_pivot.columns:
        # Drop NaNs to ensure we only have neurons present on both days
        valid_pairs = df_pivot[[day, next_day]].dropna()
        rois.update(valid_pairs.index)
        x_vals.extend(valid_pairs[day].values)
        y_vals.extend(valid_pairs[next_day].values)
        print(f'{len(valid_pairs)} for day {day}')

# 3. Convert to cm for easier reading
x_vals = np.array(x_vals) * 100
y_vals = np.array(y_vals) * 100
print(f'Total: {len(x_vals)} pairs from {len(rois)} neurons')
# 4. Plot
fig, ax = plt.subplots(figsize=(3, 3))

# Use log-log scale because depth is usually distributed logarithmically
ax.scatter(x_vals, y_vals, alpha=0.5, s=20, color='royalblue', edgecolors='none')

# Add unity line
min_val = min(x_vals.min(), y_vals.min())
max_val = max(x_vals.max(), y_vals.max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Unity')

# Formatting
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Preferred Depth Day $N$ (cm)')
ax.set_ylabel('Preferred Depth Day $N+1$ (cm)')


# Add correlation coefficient
from scipy.stats import spearmanr
# Calculate correlation in log space
rho, p = spearmanr(x_vals, y_vals)
# Update the text on the plot
ax.text(0.05, 0.85, f'Spearman $\\rho = {rho:.2f}$\n$p = {p:.2e}$', 
        transform=ax.transAxes, fontsize=6)

plt.tight_layout()

sns.despine(ax=ax, offset=3, trim=False)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
fig.savefig(save_folder / "consecutive_day_tuning.pdf", 
    format='pdf', 
    bbox_inches='tight',   # Automatically crops whitespace bounds
    transparent=False)  


In [ ]:
print(depth_tuning.shape)
logdepth_tuning = np.log(depth_tuning)
avg_d = np.nanmedian(logdepth_tuning, axis=1)
np.nanmin(logdepth_tuning), np.nanmax(logdepth_tuning)

In [ ]:
valid = np.all(np.nan_to_num(is_depth_tuned, nan=1)==1, axis=1)


tuning_change = logdepth_tuning - avg_d[:, None]
d =tuning_change[valid,:]
plt.imshow(d[np.argsort(depth_tuning[valid,:][:,0]),:], vmin=-2, vmax=2, cmap='RdBu', aspect='auto',interpolation='None')